In [ ]:
!pip install flask fpdf pyngrok --quiet


In [7]:
import os

# Create folders
os.makedirs("templates", exist_ok=True)

# app.py
with open("app.py", "w") as f:
    f.write('''from flask import Flask, render_template, request, send_file
from excuse_generator import generate_excuse
from proof_generator import generate_fake_doctors_note

app = Flask(__name__)

@app.route('/', methods=['GET', 'POST'])
def index():
    excuse = ""
    if request.method == 'POST':
        scenario = request.form.get('scenario')
        tone = request.form.get('tone')
        action = request.form.get('action')

        if action == "generate_excuse":
            excuse = generate_excuse(scenario, tone)
        elif action == "generate_note":
            generate_fake_doctors_note("John Doe")
            return send_file("doctors_note.pdf", as_attachment=True)

    return render_template('index.html', excuse=excuse)

@app.route('/chat')
def chat():
    return render_template('fake_chat.html')

if __name__ == '__main__':
    app.run()
''')

# excuse_generator.py
with open("excuse_generator.py", "w") as f:
    f.write('''import random

templates = {
    "work": ["I’m terribly sorry, I can’t make it to work today due to {reason}.",
             "Something urgent came up, and I need to miss work: {reason}.",
             "Please excuse my absence — I'm dealing with {reason}."],
    "school": ["I can't attend class because of {reason}.",
               "I’m unable to submit the assignment due to {reason}.",
               "Please excuse my absence from school today — {reason} happened."],
    "social": ["I'm really sorry, but I can't come to the event due to {reason}.",
               "Hope you understand — I’m caught up with {reason}.",
               "Won’t make it tonight, something came up: {reason}."],
    "family": ["Family emergency: I need to deal with {reason}.",
               "Sorry, my family needs me right now due to {reason}.",
               "I have to be with my family — {reason} just happened."]
}

reasons = ["a severe migraine", "a last-minute family emergency", "urgent mental health needs", "a car breakdown"]

tones = {
    "emotional": "I feel terrible having to say this, but ",
    "professional": "Please understand that ",
    "neutral": ""
}

def generate_excuse(scenario="work", tone="neutral"):
    scenario = scenario.lower()
    tone_prefix = tones.get(tone.lower(), "")
    if scenario not in templates:
        return "Invalid scenario. Choose from: work, school, social, family."
    excuse_template = random.choice(templates[scenario])
    reason = random.choice(reasons)
    return tone_prefix + excuse_template.format(reason=reason)
''')

# proof_generator.py
with open("proof_generator.py", "w") as f:
    f.write('''from fpdf import FPDF
import datetime
import random

def generate_fake_doctors_note(patient_name="John Doe", output_file="doctors_note.pdf"):
    date_today = datetime.datetime.now().strftime("%B %d, %Y")
    rest_days = random.randint(1, 3)

    note = FPDF()
    note.add_page()
    note.set_font("Arial", size=12)
    note.cell(200, 10, txt="Official Medical Excuse Note", ln=1, align="C")
    note.ln(10)
    note.multi_cell(0, 10, f"This note certifies that {patient_name} visited our clinic on {date_today}. "
                           f"The patient is advised to rest for {rest_days} day(s).")
    note.ln(20)
    note.cell(0, 10, txt="Sincerely,", ln=1)
    note.cell(0, 10, txt="Dr. Alex Thompson, M.D.", ln=1)
    note.output(output_file)
    return output_file
''')

# templates/index.html
with open("templates/index.html", "w") as f:
    f.write('''<!DOCTYPE html>
<html>
<head><title>Excuse Generator</title></head>
<body style="font-family: Arial; text-align: center;">
    <h2>🤖 Intelligent Excuse Generator</h2>
    <form method="POST">
        <label>Scenario:</label>
        <select name="scenario">
            <option value="work">Work</option>
            <option value="school">School</option>
            <option value="social">Social</option>
            <option value="family">Family</option>
        </select><br><br>
        <label>Tone:</label>
        <select name="tone">
            <option value="neutral">Neutral</option>
            <option value="emotional">Emotional</option>
            <option value="professional">Professional</option>
        </select><br><br>
        <button type="submit" name="action" value="generate_excuse">Generate Excuse</button>
        <button type="submit" name="action" value="generate_note">Download Doctor's Note</button>
    </form><br>
    <a href="/chat">View Fake Chat</a><br><br>
    {% if excuse %}
    <div style="margin-top: 20px; font-weight: bold;">{{ excuse }}</div>
    {% endif %}
</body>
</html>
''')

# templates/fake_chat.html
with open("templates/fake_chat.html", "w") as f:
    f.write('''<!DOCTYPE html>
<html>
<head><title>Fake Chat</title></head>
<body style="font-family: Arial; padding: 20px; background: #eee;">
    <div style="max-width: 300px; margin: auto; background: white; padding: 10px; border-radius: 10px;">
        <div style="background: #ddd; padding: 10px; margin: 5px; border-radius: 8px;">Are you coming?</div>
        <div style="background: #cfc; padding: 10px; margin: 5px; border-radius: 8px;">Sorry, emergency at home.</div>
        <div style="background: #ddd; padding: 10px; margin: 5px; border-radius: 8px;">Hope everything is okay!</div>
    </div>
</body>
</html>
''')


In [ ]:
from pyngrok import ngrok
import threading
import time
import os
from pyngrok import ngrok

ngrok.set_auth_token("2wZLH0j36kyrczijVM4hmgdydYx_72kmNCjyDu6AeYcVeyz2x")
print("✅ Ngrok authtoken set successfully.")


# Connect ngrok to port 5000
public_url = ngrok.connect(5000)
print("🚀 Your public URL:", public_url)

# Run Flask app in background
def run_flask():
    os.system("python app.py")

thread = threading.Thread(target=run_flask)
thread.start()

# Keep session alive
while True:
    time.sleep(10)


✅ Ngrok authtoken set successfully.
🚀 Your public URL: NgrokTunnel: "https://a32d-34-106-114-151.ngrok-free.app" -> "http://localhost:5000"
